In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)

/home/yingte/anaconda3/envs/world/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/yingte/anaconda3/envs/world/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/fla-hub/rwkv7-2.9B-g1:
- modeling_rwkv7.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/fla-hub/rwkv7-2.9B-g1:
- hf_rwkv_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of 

In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="BlinkDL/rwkv7-g1",
    filename="rwkv7-g1c-2.9b-20251231-ctx8192.pth",
)
print(f"Downloaded to: {model_path}")

Downloaded to: /home/yingte/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/bf5ef909cee14aafad0a79987dca7fbb922da57d/rwkv7-g1c-2.9b-20251231-ctx8192.pth


In [6]:
import torch

# Load the model checkpoint
model_state = torch.load(model_path, map_location='cpu', weights_only=True)
print(f"Model loaded successfully!")
print(f"Total parameters in checkpoint: {len(model_state)}")

# Infer model configuration from checkpoint
# 1. n_embd: from embedding layer shape
n_embd = model_state['emb.weight'].shape[1]
print(f"\nn_embd (embedding dimension): {n_embd}")

# 2. vocab_size: from embedding layer shape  
vocab_size = model_state['emb.weight'].shape[0]
print(f"vocab_size: {vocab_size}")

# 3. n_layer: count unique block indices
layer_keys = [k for k in model_state.keys() if k.startswith('blocks.')]
n_layer = max([int(k.split('.')[1]) for k in layer_keys]) + 1
print(f"n_layer: {n_layer}")

# 4. head_size_a: from ln_x GroupNorm num_groups or r_k shape
r_k_key = [k for k in model_state.keys() if 'r_k' in k][0]
head_size_a = model_state[r_k_key].shape[1]
n_head = model_state[r_k_key].shape[0]
print(f"head_size_a: {head_size_a}")
print(f"n_head: {n_head}")

# 5. LoRA dimensions from weight shapes
w1_key = [k for k in model_state.keys() if '.w1' in k][0]
D_DECAY_LORA = model_state[w1_key].shape[1]
print(f"D_DECAY_LORA: {D_DECAY_LORA}")

a1_key = [k for k in model_state.keys() if '.a1' in k][0]
D_AAA_LORA = model_state[a1_key].shape[1]
print(f"D_AAA_LORA: {D_AAA_LORA}")

v1_key = [k for k in model_state.keys() if '.v1' in k][0]
D_MV_LORA = model_state[v1_key].shape[1]
print(f"D_MV_LORA: {D_MV_LORA}")

g1_key = [k for k in model_state.keys() if '.g1' in k][0]
D_GATE_LORA = model_state[g1_key].shape[1]
print(f"D_GATE_LORA: {D_GATE_LORA}")

# Show some example parameter shapes
print("\nSample parameter shapes:")
for k in list(model_state.keys())[:15]:
    print(f"  {k}: {model_state[k].shape}")

Model loaded successfully!
Total parameters in checkpoint: 1062

n_embd (embedding dimension): 2560
vocab_size: 65536
n_layer: 32
head_size_a: 64
n_head: 40
D_DECAY_LORA: 96
D_AAA_LORA: 96
D_MV_LORA: 64
D_GATE_LORA: 320

Sample parameter shapes:
  emb.weight: torch.Size([65536, 2560])
  blocks.0.ln1.weight: torch.Size([2560])
  blocks.0.ln1.bias: torch.Size([2560])
  blocks.0.ln2.weight: torch.Size([2560])
  blocks.0.ln2.bias: torch.Size([2560])
  blocks.0.ln0.weight: torch.Size([2560])
  blocks.0.ln0.bias: torch.Size([2560])
  blocks.0.att.x_r: torch.Size([1, 1, 2560])
  blocks.0.att.x_w: torch.Size([1, 1, 2560])
  blocks.0.att.x_k: torch.Size([1, 1, 2560])
  blocks.0.att.x_v: torch.Size([1, 1, 2560])
  blocks.0.att.x_a: torch.Size([1, 1, 2560])
  blocks.0.att.x_g: torch.Size([1, 1, 2560])
  blocks.0.att.w0: torch.Size([1, 1, 2560])
  blocks.0.att.r_k: torch.Size([40, 64])


In [ ]:
import sys
import types
import torch.nn as nn
from torch.nn import functional as F

# Add RWKV-v7 to path for imports
sys.path.insert(0, '/home/yingte/projects/RWKV-LM/RWKV-v7')

########################################################################################################
# Model Configuration (inferred from checkpoint in cell 3)
########################################################################################################

args = types.SimpleNamespace()
args.n_layer = n_layer          # 32 layers
args.n_embd = n_embd            # 2560 embedding dimension
args.vocab_size = vocab_size    # 65536 tokens
args.head_size_a = head_size_a  # 64 (standard for RWKV-7)

# LoRA dimensions (inferred from checkpoint weights)
# These control the low-rank decomposition in various components
# Formula: D_XXX_LORA scales roughly with model size
D_DECAY_LORA = D_DECAY_LORA     # 96 - for time decay computation
D_AAA_LORA = D_AAA_LORA         # 96 - for "in-context learning rate" 
D_MV_LORA = D_MV_LORA           # 64 - for value residual mixing
D_GATE_LORA = D_GATE_LORA       # 320 - for gating mechanism

HEAD_SIZE = args.head_size_a
DTYPE = torch.half  # Use float16 for efficiency

# Disable CUDA kernel (use pure PyTorch for portability)
USE_CUDA_KERNEL = False

# JIT settings from rwkv_v7_demo.py
MyModule = nn.Module  # Use regular nn.Module instead of ScriptModule for flexibility
MyFunction = lambda x: x  # No-op decorator
MyStatic = lambda x: x

print("Model Configuration (inferred from checkpoint):")
print(f"  n_layer: {args.n_layer}")
print(f"  n_embd: {args.n_embd}")
print(f"  vocab_size: {args.vocab_size}")
print(f"  head_size_a: {args.head_size_a}")
print(f"  n_head: {args.n_embd // args.head_size_a}")
print(f"  D_DECAY_LORA: {D_DECAY_LORA}")
print(f"  D_AAA_LORA: {D_AAA_LORA}")
print(f"  D_MV_LORA: {D_MV_LORA}")
print(f"  D_GATE_LORA: {D_GATE_LORA}")
print(f"\nHow configurations were determined:")
print(f"  - n_embd: emb.weight.shape[1] = {n_embd}")
print(f"  - vocab_size: emb.weight.shape[0] = {vocab_size}")
print(f"  - n_layer: max block index + 1 = {n_layer}")
print(f"  - head_size_a: blocks.0.att.r_k.shape[1] = {head_size_a}")
print(f"  - D_*_LORA: from respective weight matrices (w1, a1, v1, g1)")

In [ ]:
########################################################################################################
# RWKV-7 Model Architecture (from RWKV-v7/rwkv_v7_demo.py)
# Using pure PyTorch implementation (no CUDA kernel) for portability
########################################################################################################

def RWKV7_OP(r, w, k, v, a, b):
    """
    RWKV-7 time-mixing operation - the core recurrent computation.
    Pure PyTorch implementation from rwkv_v7_demo.py (line 170-190)
    """
    B, T, C = r.size()
    H = C // HEAD_SIZE
    N = HEAD_SIZE
    r = r.view(B, T, H, N).float()
    k = k.view(B, T, H, N).float()
    v = v.view(B, T, H, N).float()
    a = a.view(B, T, H, N).float()
    b = b.view(B, T, H, N).float()
    w = torch.exp(-torch.exp(w.view(B, T, H, N).float()))
    out = torch.zeros((B, T, H, N), device=r.device, dtype=torch.float)
    state = torch.zeros((B, H, N, N), device=r.device, dtype=torch.float)

    for t in range(T):
        kk = k[:, t, :].view(B, H, 1, N)
        rr = r[:, t, :].view(B, H, N, 1)
        vv = v[:, t, :].view(B, H, N, 1)
        aa = a[:, t, :].view(B, H, N, 1)
        bb = b[:, t, :].view(B, H, 1, N)
        state = state * w[:, t, :, None, :] + state @ aa @ bb + vv @ kk
        out[:, t, :] = (state @ rr).view(B, H, N)

    return out.view(B, T, C).to(dtype=DTYPE)


class RWKV_Tmix_x070(MyModule):
    """RWKV-7 Time Mixing (attention-like) - from rwkv_v7_demo.py line 207-277"""
    def __init__(self, args, layer_id):
        super().__init__()
        self.args = args
        self.layer_id = layer_id
        self.head_size = args.head_size_a
        self.n_head = args.dim_att // self.head_size
        assert args.dim_att % self.n_head == 0

        H = self.n_head
        N = self.head_size
        C = args.n_embd

        self.x_r = nn.Parameter(torch.empty(1, 1, C))
        self.x_w = nn.Parameter(torch.empty(1, 1, C))
        self.x_k = nn.Parameter(torch.empty(1, 1, C))
        self.x_v = nn.Parameter(torch.empty(1, 1, C))
        self.x_a = nn.Parameter(torch.empty(1, 1, C))
        self.x_g = nn.Parameter(torch.empty(1, 1, C))

        self.w0 = nn.Parameter(torch.empty(1, 1, C))
        self.w1 = nn.Parameter(torch.empty(C, D_DECAY_LORA))
        self.w2 = nn.Parameter(torch.empty(D_DECAY_LORA, C))

        self.a0 = nn.Parameter(torch.empty(1, 1, C))
        self.a1 = nn.Parameter(torch.empty(C, D_AAA_LORA))
        self.a2 = nn.Parameter(torch.empty(D_AAA_LORA, C))

        self.v0 = nn.Parameter(torch.empty(1, 1, C))
        self.v1 = nn.Parameter(torch.empty(C, D_MV_LORA))
        self.v2 = nn.Parameter(torch.empty(D_MV_LORA, C))

        self.g1 = nn.Parameter(torch.empty(C, D_GATE_LORA))
        self.g2 = nn.Parameter(torch.empty(D_GATE_LORA, C))

        self.k_k = nn.Parameter(torch.empty(1, 1, C))
        self.k_a = nn.Parameter(torch.empty(1, 1, C))
        self.r_k = nn.Parameter(torch.empty(H, N))

        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))
        self.receptance = nn.Linear(C, C, bias=False)
        self.key = nn.Linear(C, C, bias=False)
        self.value = nn.Linear(C, C, bias=False)
        self.output = nn.Linear(C, C, bias=False)
        self.ln_x = nn.GroupNorm(H, C, eps=64e-5)

    def forward(self, x, v_first):
        B, T, C = x.size()
        H = self.n_head
        xx = self.time_shift(x) - x

        xr = x + xx * self.x_r
        xw = x + xx * self.x_w
        xk = x + xx * self.x_k
        xv = x + xx * self.x_v
        xa = x + xx * self.x_a
        xg = x + xx * self.x_g

        r = self.receptance(xr)
        w = -F.softplus(-(self.w0 + torch.tanh(xw @ self.w1) @ self.w2)) - 0.5
        k = self.key(xk)
        v = self.value(xv)
        if self.layer_id == 0:
            v_first = v
        else:
            v = v + (v_first - v) * torch.sigmoid(self.v0 + (xv @ self.v1) @ self.v2)
        a = torch.sigmoid(self.a0 + (xa @ self.a1) @ self.a2)
        g = torch.sigmoid(xg @ self.g1) @ self.g2

        kk = k * self.k_k
        kk = F.normalize(kk.view(B, T, H, -1), dim=-1, p=2.0).view(B, T, C)
        k = k * (1 + (a - 1) * self.k_a)

        x = RWKV7_OP(r, w, k, v, -kk, kk * a)
        x = self.ln_x(x.view(B * T, C)).view(B, T, C)
        
        x = x + ((r.view(B, T, H, -1) * k.view(B, T, H, -1) * self.r_k).sum(dim=-1, keepdim=True) * v.view(B, T, H, -1)).view(B, T, C)
        x = self.output(x * g)
        return x, v_first


class RWKV_CMix_x070(MyModule):
    """RWKV-7 Channel Mixing (FFN-like) - from rwkv_v7_demo.py line 283-299"""
    def __init__(self, args, layer_id):
        super().__init__()
        self.args = args
        self.layer_id = layer_id
        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))

        self.x_k = nn.Parameter(torch.empty(1, 1, args.n_embd))
        self.key = nn.Linear(args.n_embd, args.dim_ffn, bias=False)
        self.value = nn.Linear(args.dim_ffn, args.n_embd, bias=False)

    def forward(self, x):
        xx = self.time_shift(x) - x
        k = x + xx * self.x_k
        k = torch.relu(self.key(k)) ** 2
        return self.value(k)


class Block(MyModule):
    """RWKV-7 Block - from rwkv_v7_demo.py line 305-325"""
    def __init__(self, args, layer_id):
        super().__init__()
        self.args = args
        self.layer_id = layer_id

        self.ln0 = nn.LayerNorm(args.n_embd)
        self.ln1 = nn.LayerNorm(args.n_embd)
        self.ln2 = nn.LayerNorm(args.n_embd)

        self.att = RWKV_Tmix_x070(args, layer_id)
        self.ffn = RWKV_CMix_x070(args, layer_id)

    def forward(self, x, v_first):
        if self.layer_id == 0:
            x = self.ln0(x)
        xx, v_first = self.att(self.ln1(x), v_first)
        x = x + xx
        x = x + self.ffn(self.ln2(x))
        return x, v_first


class RWKV(nn.Module):
    """Full RWKV-7 Model - from rwkv_v7_demo.py line 331-372"""
    def __init__(self, args):
        super().__init__()
        args.dim_att = args.n_embd
        args.dim_ffn = args.n_embd * 4
        
        self.emb = nn.Embedding(args.vocab_size, args.n_embd)
        self.blocks = nn.ModuleList([Block(args, i) for i in range(args.n_layer)])
        self.ln_out = nn.LayerNorm(args.n_embd)
        self.head = nn.Linear(args.n_embd, args.vocab_size, bias=False)

    def forward(self, idx):
        x = self.emb(idx)
        v_first = torch.empty_like(x)
        for block in self.blocks:
            x, v_first = block(x, v_first)
        x = self.ln_out(x)
        x = self.head(x)
        return x

print("RWKV-7 model architecture loaded (based on RWKV-v7/rwkv_v7_demo.py)")

In [ ]:
########################################################################################################
# Initialize and Load Model
########################################################################################################

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

with torch.no_grad():
    rwkv_model = RWKV(args).to(dtype=DTYPE).to(device)
    rwkv_model.load_state_dict(model_state, strict=False)
    # strict=False: blocks.0.att.v0/v1/v2 exist in checkpoint but not used in layer 0

print(f"\n✓ Model loaded successfully!")
print(f"  Total parameters: {sum(p.numel() for p in rwkv_model.parameters()):,}")
print(f"  Model dtype: {DTYPE}")
print(f"  Device: {device}")

In [ ]:
########################################################################################################
# Test Inference
########################################################################################################

prompt = "The Eiffel tower is in the city of"
input_ids = tokenizer.encode(prompt)
print(f"Prompt: {prompt}")
print(f"Tokens: {len(input_ids)}")

with torch.no_grad():
    input_tensor = torch.tensor(input_ids).reshape(1, -1).to(device)
    out = rwkv_model.forward(input_tensor)
    
    logits = out[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    
    print(f"\nTop 10 predictions for next token:")
    _, indices = torch.topk(probs, 10)
    for i, idx in enumerate(indices):
        token_id = idx.item()
        token = tokenizer.decode([token_id])
        prob = probs[token_id].item()
        print(f"  {i+1}. '{token}' ({prob:.2%})")